
# Synthetic attribution experiment — calibrated stress test

Цель эксперимента — **не доказать, что одна rule-based attribution-модель является “истиной”**.  
Мы хотим проверить более практичный вопрос:

> насколько `first / last / linear / position` правильно ранжируют рекламные размещения по эффективности, если основной объём продаж создаёт естественный/сезонный спрос, а реклама даёт меньший incremental lift?

Ключевая идея генератора:

1. Исторические данные `base.csv / base.xlsx` используются для калибровки:
   - распределения чеков;
   - структуры заказов и пакетов;
   - доли повторных покупателей;
   - временного профиля спроса.
2. Сначала возникает **organic demand**.
3. Только если органическая покупка не произошла, реклама может создать **дополнительную** покупку.
4. Истинный драйвер рекламной покупки (`true_driver`) сохраняется только в hidden ground truth и **не передаётся attribution engine**.
5. Рекламные размещения специально ставятся рядом с периодами высокого спроса. Это создаёт реалистичную ловушку: attribution может приписать рекламе продажи, которые случились бы и без неё.

Если ноутбук запускается внутри репозитория, для расчёта используется текущий `src/core/attribution.py`.  
Вне репозитория включается компактный fallback с теми же четырьмя правилами распределения — только для воспроизводимого preview.



## 0. Запуск в Google Colab

Эта версия ноутбука умеет запускаться прямо из GitHub через Colab.

Если ноутбук открыт в Colab:

1. репозиторий автоматически клонируется в `/content/hackaton_project`;
2. устанавливаются зависимости из `requirements.txt`, если файл существует;
3. рабочая директория переключается на корень репозитория;
4. `src/` добавляется в `sys.path`;
5. исторические данные читаются из репозитория;
6. расчёт attribution выполняется через текущий `src/core/attribution.py`.

Если ноутбук запускается локально внутри репозитория, повторное клонирование не выполняется.


In [ ]:

# --- Colab / local bootstrap ---
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/valentinesvev/hackaton_project.git"
REPO_DIR = Path("/content/hackaton_project")

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Repository already exists:", REPO_DIR)

    os.chdir(REPO_DIR)

    requirements = REPO_DIR / "requirements.txt"
    if requirements.exists():
        print("Installing requirements...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
            check=True
        )
    else:
        print("requirements.txt not found — using current environment.")

    ROOT = REPO_DIR
else:
    # Локальный запуск: определяем корень репозитория.
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

src_dir = ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print("Working directory:", Path.cwd())
print("Project root:", ROOT)
print("src in sys.path:", src_dir)


In [ ]:

from pathlib import Path
import sys
import math
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
N_USERS = 1000
N_RUNS = 30
WINDOW_DAYS = 30
ACQUIRING_RATE = 0.03
DECAY_DAYS = 7.0

# ROOT is set by the Colab/local bootstrap cell above.

OUTPUT_DIR = ROOT / "outputs" / "charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Если ноутбук лежит в репозитории — используем production attribution.py.
USE_PRODUCTION_ENGINE = False
try:
    from core.db import connect
    from core import attribution
    USE_PRODUCTION_ENGINE = True
    print("Engine: production src/core/attribution.py")
except Exception as exc:
    print("Engine: notebook fallback (production module не найден в текущем окружении)")
    print("Reason:", type(exc).__name__, str(exc)[:200])



## 1. Калибровка по историческим продажам

Сначала собираем **заказы**, а не считаем каждую строку исходного файла отдельной покупкой.

Правило нормализации для базового эксперимента:

- строки одного `student_id` с **одинаковым timestamp** считаем одним заказом;
- сумма заказа = сумма строк;
- несколько курсов внутри такого заказа считаем bundle;
- повторный покупатель = пользователь с более чем одним distinct order.

Это приближение явно фиксируем как assumption.


In [ ]:

def find_history_file(root: Path):
    candidates = [
        root / "data" / "raw" / "base.csv",
        root / "data" / "raw" / "base.xlsx",
        root / "data" / "base.csv",
        root / "data" / "base.xlsx",
        root / "base.csv",
        root / "base.xlsx",
        Path("/mnt/data/base.csv"),
        Path("/mnt/data/base.xlsx"),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def load_history(root=ROOT):
    path = find_history_file(root)

    if path is None:
        # Удобно при запуске ноутбука прямо из GitHub/Colab.
        url = "https://raw.githubusercontent.com/valentinesvev/hackaton_project/main/data/raw/base.csv"
        try:
            raw = pd.read_csv(url)
            source = url
        except Exception:
            return None, None
    else:
        source = str(path)
        raw = pd.read_excel(path) if path.suffix.lower() == ".xlsx" else pd.read_csv(path)

    # Поддерживаем русские имена из исходного base и английские варианты.
    rename_map = {
        "Номер студента": "student_id",
        "Сумма": "amount",
        "Курс": "course",
        "Время": "timestamp",
    }
    raw = raw.rename(columns=rename_map).copy()

    if "amount" in raw:
        raw["amount"] = (
            raw["amount"].astype(str)
            .str.replace(r"\s", "", regex=True)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )

    raw["timestamp"] = pd.to_datetime(raw["timestamp"], dayfirst=True, errors="coerce")
    raw = raw.dropna(subset=["student_id", "amount", "timestamp"]).copy()

    # Одновременные строки одного пользователя -> один order/bundle.
    orders = (
        raw.groupby(["student_id", "timestamp"], as_index=False)
        .agg(
            amount=("amount", "sum"),
            n_items=("course", "size"),
            courses=("course", lambda x: tuple(x.astype(str))),
        )
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    orders["order_id"] = np.arange(1, len(orders) + 1)

    return raw, orders


history_raw, history_orders = load_history()

if history_orders is not None:
    repeat_rate = (
        history_orders.groupby("student_id").size().gt(1).mean()
    )
    bundle_rate = history_orders["n_items"].gt(1).mean()

    daily = (
        history_orders.assign(day=history_orders["timestamp"].dt.normalize())
        .groupby("day").size()
        .rename("orders")
        .asfreq("D", fill_value=0)
    )
    # Сглаживаем случайный дневной шум, но сохраняем форму пиков.
    demand_profile = daily.rolling(5, center=True, min_periods=1).mean()
    demand_index = demand_profile / demand_profile.mean()

    amount_pool = history_orders["amount"].to_numpy(float)

    print("History source:", find_history_file(ROOT) or "GitHub raw/base.csv")
    print("Normalized orders:", len(history_orders))
    print("Unique buyers:", history_orders["student_id"].nunique())
    print("Bundle share:", round(bundle_rate, 3))
    print("Repeat-buyer share:", round(repeat_rate, 3))
    print("Median order:", round(float(np.median(amount_pool)), 0), "₽")
else:
    # Fallback только для preview, если исторический файл недоступен.
    repeat_rate = 0.20
    bundle_rate = 0.18
    amount_pool = np.array([4950, 6490, 6950, 8245, 8745, 8950, 9950, 14950, 16950], dtype=float)

    idx = pd.date_range("2026-08-04", "2026-09-10", freq="D")
    x = np.arange(len(idx))
    profile = (
        0.65
        + 1.8 * np.exp(-0.5 * ((x - 5) / 2.8) ** 2)
        + 1.1 * np.exp(-0.5 * ((x - 31) / 3.2) ** 2)
    )
    demand_index = pd.Series(profile / profile.mean(), index=idx, name="demand_index")

    print("Исторический файл недоступен: используется fallback-калибровка только для preview.")


In [ ]:

# Визуализация профиля спроса, который используется генератором.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(demand_index.index, demand_index.values, marker="o", markersize=3)
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_title("Исторический профиль спроса → baseline synthetic-генератора")
ax.set_ylabel("Индекс спроса (среднее = 1)")
ax.set_xlabel("Дата")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

DEMAND_CHART = OUTPUT_DIR / "synthetic_demand_profile.png"
fig.savefig(DEMAND_CHART, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", DEMAND_CHART)



## 2. Рекламные размещения и скрытый causal ground truth

У каждого placement есть:

- `publication_time` и `cost`, которые видит рабочая система;
- `reach` и `true_daily_lift`, которые существуют **только внутри генератора**.

`true_daily_lift` — небольшая дополнительная вероятность покупки после рекламного касания.  
Она экспоненциально затухает по времени.

Важно: размещения стоят рядом с периодами высокого естественного спроса. Поэтому даже слабая реклама может оказаться “последним касанием” перед большим органическим всплеском.


In [ ]:

SIM_START = pd.Timestamp(demand_index.index.min())
SIM_END = pd.Timestamp(demand_index.index.max())

# D специально публикуется около позднего пика: большой reach, но слабый true lift.
# Это стресс-тест для last touch.
placements = pd.DataFrame([
    {
        "placement_id": "a", "channel": "tg_ml_jobs", "campaign_name": "autumn_2026",
        "placement_type": "external", "creative": "creative_a", "post_category": "sale",
        "discount_value": 0, "cost": 70000,
        "publication_time": SIM_START + pd.Timedelta(days=3),
        "reach": 0.24, "true_daily_lift": 0.0060,
    },
    {
        "placement_id": "b", "channel": "tg_data_jobs", "campaign_name": "autumn_2026",
        "placement_type": "external", "creative": "creative_b", "post_category": "native",
        "discount_value": 0, "cost": 55000,
        "publication_time": SIM_START + pd.Timedelta(days=13),
        "reach": 0.22, "true_daily_lift": 0.0048,
    },
    {
        "placement_id": "c", "channel": "tg_students", "campaign_name": "autumn_2026",
        "placement_type": "external", "creative": "creative_c", "post_category": "discount",
        "discount_value": 15, "cost": 40000,
        "publication_time": SIM_START + pd.Timedelta(days=23),
        "reach": 0.18, "true_daily_lift": 0.0035,
    },
    {
        "placement_id": "d", "channel": "tg_career", "campaign_name": "autumn_2026",
        "placement_type": "external", "creative": "creative_d", "post_category": "content",
        "discount_value": 0, "cost": 40000,
        "publication_time": SIM_START + pd.Timedelta(days=30),
        "reach": 0.34, "true_daily_lift": 0.0015,
    },
])

display(
    placements[
        ["placement_id", "channel", "publication_time", "cost", "reach", "true_daily_lift"]
    ]
)


PLACEMENT_LOOKUP = placements.set_index("placement_id").to_dict("index")



## 3. Генератор

### Покупка возникает в два этапа

Для каждого user-day:

1. С вероятностью `p_organic` происходит **органическая** покупка.
   - `p_organic` зависит от исторического demand index;
   - у пользователей есть латентная склонность к покупке.
2. Только если органическая покупка не произошла, реклама может дать дополнительную покупку.
   - учитываются только рекламные touches после предыдущей покупки;
   - эффект касания затухает;
   - если рекламная покупка произошла, `true_driver` выбирается пропорционально скрытому эффекту placement.

Так мы точно знаем, какая продажа **добавлена рекламой**, а какая произошла бы и без неё.

Это специально отличается от rule-based attribution: production engine не знает `true_driver`.


In [ ]:

def demand_at(day):
    day = pd.Timestamp(day).normalize()
    if day in demand_index.index:
        return float(demand_index.loc[day])

    # На случай несовпадения индексов — ближайший день.
    pos = demand_index.index.get_indexer([day], method="nearest")[0]
    return float(demand_index.iloc[pos])


def decayed_effect(pid, age_days):
    if age_days < 0 or age_days > WINDOW_DAYS:
        return 0.0
    return float(PLACEMENT_LOOKUP[pid]["true_daily_lift"]) * math.exp(-age_days / DECAY_DAYS)


def sample_amount(rng):
    return float(rng.choice(amount_pool))


def eligible_paid_touches(user_touches, day, prev_order_ts=None):
    out = []
    day = pd.Timestamp(day)

    for t in user_touches:
        pid = t["placement_id"]
        if pid is None:
            continue

        ts = pd.Timestamp(t["timestamp"])
        age = (day - ts).total_seconds() / 86400

        if 0 <= age <= WINDOW_DAYS:
            if prev_order_ts is None or ts > pd.Timestamp(prev_order_ts):
                out.append((t, age))

    return out


def generate_synthetic(seed=SEED, n_users=N_USERS):
    rng = np.random.default_rng(seed)
    days = pd.date_range(SIM_START, SIM_END, freq="D")

    users, touches, events, leads, orders, order_items = [], [], [], [], [], []
    order_truth, user_truth = [], []

    touch_id = 1
    event_id = 1
    lead_id = 1
    item_id = 1

    # Калибруем среднюю дневную organic probability так,
    # чтобы за весь период значимая доля пользователей могла купить.
    base_daily_purchase = 0.018

    for i in range(n_users):
        uid = f"u{i:05d}"

        # Латентная склонность: часть пользователей заметно "горячее" остальных.
        propensity = float(rng.lognormal(mean=-0.08, sigma=0.45))
        propensity = float(np.clip(propensity, 0.45, 2.4))

        user_touches = []

        def add_touch(ts, pid):
            nonlocal touch_id
            row = {
                "touch_id": touch_id,
                "user_id_hash": uid,
                "placement_id": pid,
                "touch_type": "bot_start",
                "timestamp": pd.Timestamp(ts),
                "is_synthetic": 1,
            }
            touches.append(row)
            user_touches.append(row)
            touch_id += 1
            return row

        # Mild selection: более "горячие" пользователи чуть чаще кликают рекламу.
        for p in placements.itertuples():
            click_prob = min(0.90, p.reach * (0.8 + 0.2 * propensity))
            if rng.random() < click_prob:
                lag_hours = int(rng.integers(0, 60))
                add_touch(pd.Timestamp(p.publication_time) + pd.Timedelta(hours=lag_hours), p.placement_id)

        # Часть пользователей приходит в бот органически.
        if rng.random() < 0.28:
            organic_ts = SIM_START + pd.Timedelta(
                days=int(rng.integers(0, max(1, (SIM_END - SIM_START).days + 1))),
                hours=int(rng.integers(8, 22)),
            )
            add_touch(organic_ts, None)

        if not user_touches:
            # Пользователь всё равно существует в dim_user, даже если в бот не заходил.
            first_seen = SIM_START + pd.Timedelta(days=int(rng.integers(0, 5)))
            first_pid = None
        else:
            user_touches.sort(key=lambda r: r["timestamp"])
            first_seen = user_touches[0]["timestamp"]
            first_pid = user_touches[0]["placement_id"]

        users.append({
            "user_id_hash": uid,
            "username_hash": None,
            "first_seen": first_seen,
            "first_placement_id": first_pid,
            "source_system": "bot",
        })

        user_truth.append({
            "user_id_hash": uid,
            "propensity": propensity,
            "n_ad_touches": sum(t["placement_id"] is not None for t in user_touches),
        })

        prev_order_ts = None
        order_no = 0

        # Ограничиваемся максимум двумя заказами на пользователя.
        for day in days:
            if order_no >= 2:
                break

            # После первой покупки вероятность второй покупки ниже.
            repeat_multiplier = 1.0 if order_no == 0 else min(0.55, max(0.20, repeat_rate * 1.8))

            p_org = base_daily_purchase * demand_at(day) * propensity * repeat_multiplier
            p_org = min(p_org, 0.20)

            purchase_kind = None
            true_driver = None

            # 1) Natural/seasonal demand.
            if rng.random() < p_org:
                purchase_kind = "organic"

            else:
                # 2) Small incremental ad lift.
                eligible = eligible_paid_touches(user_touches, day + pd.Timedelta(hours=18), prev_order_ts)

                effects = []
                for t, age in eligible:
                    eff = decayed_effect(t["placement_id"], age)
                    if eff > 0:
                        effects.append((t["placement_id"], eff))

                total_ad_p = min(0.12, sum(e for _, e in effects))

                if effects and rng.random() < total_ad_p:
                    purchase_kind = "ad_incremental"

                    pids = [pid for pid, _ in effects]
                    weights = np.array([e for _, e in effects], dtype=float)
                    weights /= weights.sum()
                    true_driver = rng.choice(pids, p=weights)

            if purchase_kind is None:
                continue

            # Не создаём вторую покупку слишком быстро.
            purchase_ts = pd.Timestamp(day) + pd.Timedelta(
                hours=int(rng.integers(10, 22)),
                minutes=int(rng.integers(0, 60)),
            )
            if prev_order_ts is not None and purchase_ts <= prev_order_ts + pd.Timedelta(days=3):
                continue

            order_no += 1
            amount = sample_amount(rng)

            # Привязываем lead к последнему известному touch до покупки.
            before = [t for t in user_touches if t["timestamp"] <= purchase_ts]
            lead_touch_id = max(before, key=lambda x: x["timestamp"])["touch_id"] if before else None

            leads.append({
                "lead_id": lead_id,
                "user_id_hash": uid,
                "touch_id": lead_touch_id,
                "course_interest": "synthetic_course",
                "self_reported_source": None,
                "conversation_started_at": purchase_ts - pd.Timedelta(hours=2),
                "manager_id": "synthetic_manager",
                "status": "paid",
                "is_synthetic": 1,
            })

            order_id = f"o_{uid}_{order_no}"

            orders.append({
                "order_id": order_id,
                "user_id_hash": uid,
                "lead_id": lead_id,
                "student_id": None,
                "amount": amount,
                "variable_costs": round(amount * ACQUIRING_RATE, 2),
                "variable_costs_source": "rate_estimate",
                "timestamp": purchase_ts,
                "source_system": "bot",
                "is_synthetic": 1,
            })

            order_items.append({
                "item_id": item_id,
                "order_id": order_id,
                "course": "synthetic_course",
                "amount": amount,
            })

            order_truth.append({
                "order_id": order_id,
                "user_id_hash": uid,
                "timestamp": purchase_ts,
                "amount": amount,
                "purchase_kind": purchase_kind,
                "true_driver": true_driver,
                "demand_index": demand_at(day),
            })

            lead_id += 1
            item_id += 1
            prev_order_ts = purchase_ts

            # После первой покупки некоторым пользователям создаём новое рекламное касание.
            # Так production-правило reset after purchase можно проверить на реальных данных генератора.
            if order_no == 1 and rng.random() < repeat_rate:
                future_placements = placements[
                    placements["publication_time"] > purchase_ts
                ]
                if not future_placements.empty:
                    p = future_placements.sample(
                        n=1,
                        random_state=int(rng.integers(0, 2**31 - 1)),
                    ).iloc[0]
                    add_touch(
                        max(purchase_ts + pd.Timedelta(days=4), pd.Timestamp(p["publication_time"])),
                        p["placement_id"],
                    )

        # Простые event для воронки: просмотр курса у части пользователей с touch.
        if user_touches and rng.random() < 0.72:
            t = max(user_touches, key=lambda x: x["timestamp"])
            events.append({
                "event_id": event_id,
                "user_id_hash": uid,
                "touch_id": t["touch_id"],
                "event_type": "view_course",
                "item": "synthetic_course",
                "timestamp": t["timestamp"] + pd.Timedelta(minutes=5),
                "is_synthetic": 1,
            })
            event_id += 1

    placement_db = placements.drop(columns=["reach", "true_daily_lift"]).copy()
    placement_db["created_at"] = SIM_START
    placement_db["is_synthetic"] = 1

    return {
        "dim_placement": placement_db,
        "dim_user": pd.DataFrame(users),
        "fact_touch": pd.DataFrame(touches),
        "fact_bot_event": pd.DataFrame(events),
        "fact_lead": pd.DataFrame(leads),
        "fact_order": pd.DataFrame(orders),
        "fact_order_item": pd.DataFrame(order_items),
        "order_truth": pd.DataFrame(order_truth),
        "user_truth": pd.DataFrame(user_truth),
    }


frames = generate_synthetic()

truth_preview = frames["order_truth"]["purchase_kind"].value_counts().to_frame("orders")
truth_preview["share"] = truth_preview["orders"] / truth_preview["orders"].sum()

print("users:", len(frames["dim_user"]))
print("touches:", len(frames["fact_touch"]))
print("orders:", len(frames["fact_order"]))
display(truth_preview)



## 4. Attribution engine

В production-режиме ниже вызывается **тот же код, что использует бот**.

В fallback-режиме используется локальная реализация только для того, чтобы ноутбук можно было открыть и выполнить без всего репозитория.


In [ ]:

def fallback_compute(touches, orders, model="last", window_days=30):
    o = (
        orders[["order_id", "user_id_hash", "amount", "variable_costs", "timestamp"]]
        .sort_values(["user_id_hash", "timestamp"])
        .copy()
    )
    o["timestamp"] = pd.to_datetime(o["timestamp"])
    o["prev_order_ts"] = o.groupby("user_id_hash")["timestamp"].shift()

    paid = touches[touches["placement_id"].notna()][
        ["touch_id", "user_id_hash", "placement_id", "timestamp"]
    ].copy()
    paid["timestamp"] = pd.to_datetime(paid["timestamp"])
    paid = paid.rename(columns={"timestamp": "touch_ts"})

    m = o.merge(paid, on="user_id_hash")
    m = m[
        (m["touch_ts"] <= m["timestamp"])
        & (m["touch_ts"] >= m["timestamp"] - pd.Timedelta(days=window_days))
        & (m["prev_order_ts"].isna() | (m["touch_ts"] > m["prev_order_ts"]))
    ].copy()

    m = m.sort_values(["order_id", "touch_ts", "touch_id"])
    n = m.groupby("order_id")["order_id"].transform("size")
    pos = m.groupby("order_id").cumcount()

    if model == "last":
        w = (pos == n - 1).astype(float)
    elif model == "first":
        w = (pos == 0).astype(float)
    elif model == "linear":
        w = 1 / n
    elif model == "position":
        edge = (pos == 0) | (pos == n - 1)
        w = np.select(
            [n == 1, n == 2, edge],
            [1.0, 0.5, 0.4],
            default=0.2 / (n - 2).clip(lower=1),
        )
    else:
        raise ValueError(model)

    m = m.assign(weight=w)
    m = m[m["weight"] > 0].copy()

    organic = o[~o["order_id"].isin(m["order_id"])].copy()
    organic["touch_id"] = None
    organic["placement_id"] = None
    organic["weight"] = 1.0

    out = pd.concat([m, organic], ignore_index=True)
    out["attributed_revenue"] = out["amount"] * out["weight"]
    out["attributed_margin"] = (out["amount"] - out["variable_costs"]) * out["weight"]

    return out[
        ["order_id", "touch_id", "placement_id", "weight", "attributed_revenue", "attributed_margin"]
    ]


def model_romi_from_frames(frames, window_days=WINDOW_DAYS):
    models = ("last", "first", "linear", "position")
    costs = placements.set_index("placement_id")["cost"]

    result = {}

    if USE_PRODUCTION_ENGINE:
        db_path = ROOT / "data" / "mock" / "synthetic_experiment.db"
        db_path.parent.mkdir(parents=True, exist_ok=True)
        if db_path.exists():
            db_path.unlink()

        conn = connect(str(db_path))

        for table in [
            "dim_placement", "dim_user", "fact_touch", "fact_bot_event",
            "fact_lead", "fact_order", "fact_order_item",
        ]:
            df = frames[table].copy()
            for col in df.columns:
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S")
            if len(df):
                df.to_sql(table, conn, index=False, if_exists="append")

        conn.commit()

        # Production engine: все 4 модели.
        comparison = attribution.compare_models(conn, window_days=window_days)
        comparison = comparison.reindex(costs.index)
        conn.close()
        return comparison

    for model in models:
        att = fallback_compute(
            frames["fact_touch"],
            frames["fact_order"],
            model=model,
            window_days=window_days,
        )

        rev = (
            att.dropna(subset=["placement_id"])
            .groupby("placement_id")["attributed_revenue"]
            .sum()
            .reindex(costs.index, fill_value=0)
        )
        result[model] = (rev - costs) / costs

    return pd.DataFrame(result)


model_comparison = model_romi_from_frames(frames)
display(model_comparison)



## 5. Ground truth и метрики

**True incremental revenue** placement — сумма чеков тех заказов, которые генератор создал именно благодаря рекламе и для которых этот placement был скрытым `true_driver`.

То есть:

\[
TrueROMI = \frac{IncrementalRevenue - Cost}{Cost}
\]

Это намеренно строже обычного attribution ROMI: органические продажи после рекламного касания **не считаются эффектом рекламы**.

Сравниваем модели по четырём метрикам:

- **Top-1 accuracy** — нашли ли placement с максимальным true ROMI;
- **Spearman rank correlation** — правильно ли ранжировали placements;
- **ROMI MAE** — насколько ошиблись в величине ROMI;
- **Sign accuracy** — правильно ли определили знак ROMI (`>0` / `<0`).


In [ ]:

MODELS = ("last", "first", "linear", "position")


def true_romi(order_truth):
    costs = placements.set_index("placement_id")["cost"]

    inc_rev = (
        order_truth.dropna(subset=["true_driver"])
        .groupby("true_driver")["amount"]
        .sum()
        .reindex(costs.index, fill_value=0.0)
    )

    return (inc_rev - costs) / costs


def evaluate_run(frames):
    truth = true_romi(frames["order_truth"])
    predicted = model_romi_from_frames(frames).reindex(truth.index)

    true_best = truth.idxmax()
    true_rank = truth.rank(ascending=False, method="average")

    rows = []
    for model in MODELS:
        p = predicted[model].reindex(truth.index)
        pred_rank = p.rank(ascending=False, method="average")

        rows.append({
            "model": model,
            "top1_correct": int(p.idxmax() == true_best),
            "rank_correlation": true_rank.corr(pred_rank, method="pearson"),
            "romi_mae": float((p - truth).abs().mean()),
            "sign_accuracy": float((np.sign(p) == np.sign(truth)).mean()),
            "mean_bias": float((p - truth).mean()),
        })

    return pd.DataFrame(rows), truth, predicted


one_run_score, truth, predicted = evaluate_run(frames)

comparison = pd.DataFrame({"true_incremental_romi": truth}).join(predicted)
display(comparison.sort_values("true_incremental_romi", ascending=False))
display(one_run_score)


In [ ]:

# Визуализация одного запуска: true incremental ROMI vs attribution ROMI.
plot_df = comparison.reset_index().rename(columns={"index": "placement_id"})
long = plot_df.melt(
    id_vars=["placement_id", "true_incremental_romi"],
    value_vars=list(MODELS),
    var_name="model",
    value_name="estimated_romi",
)

# Для читаемости показываем true ROMI отдельно и оценки моделей рядом как точки.
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(plot_df))
ax.scatter(x, plot_df["true_incremental_romi"], s=110, label="True incremental ROMI")

offsets = np.linspace(-0.18, 0.18, len(MODELS))
for off, model in zip(offsets, MODELS):
    ax.scatter(x + off, plot_df[model], s=55, label=model)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["placement_id"])
ax.set_ylabel("ROMI")
ax.set_xlabel("Placement")
ax.set_title("Один synthetic run: causal ground truth vs attribution estimates")
ax.legend(ncol=3)
fig.tight_layout()

ONE_RUN_CHART = OUTPUT_DIR / "synthetic_true_vs_attribution_romi.png"
fig.savefig(ONE_RUN_CHART, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", ONE_RUN_CHART)



## 6. Monte Carlo: повторяем эксперимент много раз

Один seed может быть случайно удачным или неудачным. Поэтому итог сравниваем по множеству независимых генераций.

Для хакатонного MVP достаточно `30` прогонов; при необходимости можно увеличить `N_RUNS`.


In [ ]:

def run_many(n_runs=N_RUNS, n_users=N_USERS, first_seed=1000):
    rows = []

    for seed in range(first_seed, first_seed + n_runs):
        run_frames = generate_synthetic(seed=seed, n_users=n_users)
        score, _, _ = evaluate_run(run_frames)
        score["seed"] = seed
        rows.append(score)

    return pd.concat(rows, ignore_index=True)


runs = run_many()

summary = (
    runs.groupby("model")
    .agg(
        top1_accuracy=("top1_correct", "mean"),
        mean_rank_correlation=("rank_correlation", "mean"),
        mean_romi_mae=("romi_mae", "mean"),
        sign_accuracy=("sign_accuracy", "mean"),
        mean_bias=("mean_bias", "mean"),
    )
    .sort_values(
        ["mean_rank_correlation", "top1_accuracy"],
        ascending=False,
    )
)

display(summary.style.format({
    "top1_accuracy": "{:.1%}",
    "mean_rank_correlation": "{:.3f}",
    "mean_romi_mae": "{:.3f}",
    "sign_accuracy": "{:.1%}",
    "mean_bias": "{:.3f}",
}))


In [ ]:

# Итоговая визуализация: rank correlation.
fig, ax = plt.subplots(figsize=(8, 4.5))
rank_plot = summary["mean_rank_correlation"].sort_values(ascending=False)
rank_plot.plot(kind="bar", ax=ax)
ax.set_ylim(min(-0.1, rank_plot.min() - 0.05), 1.0)
ax.set_ylabel("Mean Spearman-like rank correlation")
ax.set_xlabel("Attribution model")
ax.set_title("Насколько модель правильно ранжирует placements")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

RANK_CHART = OUTPUT_DIR / "synthetic_model_rank_correlation.png"
fig.savefig(RANK_CHART, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", RANK_CHART)


In [ ]:

# Итоговая визуализация: sign accuracy.
fig, ax = plt.subplots(figsize=(8, 4.5))
sign_plot = (summary["sign_accuracy"] * 100).sort_values(ascending=False)
sign_plot.plot(kind="bar", ax=ax)
ax.set_ylim(0, 100)
ax.set_ylabel("Correct ROMI sign, %")
ax.set_xlabel("Attribution model")
ax.set_title("Как часто модель правильно решает: placement окупается или нет")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

SIGN_CHART = OUTPUT_DIR / "synthetic_model_sign_accuracy.png"
fig.savefig(SIGN_CHART, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", SIGN_CHART)



## 7. Как интерпретировать результат

Главный вывод этого эксперимента **не должен быть заранее задан**.

Правильная логика после запуска:

1. Смотрим на `mean_rank_correlation`: какая модель стабильнее ранжирует placements.
2. Проверяем `top1_accuracy`: как часто выбран лучший placement.
3. Проверяем `sign_accuracy`: насколько надёжно решение “масштабировать / не масштабировать”.
4. Смотрим на `mean_bias` и `ROMI MAE`:
   - положительный bias означает, что attribution систематически переоценивает causal ROMI;
   - это ожидаемо, если реклама размещается рядом с сезонными пиками.

### Почему это полезно для кейса

Даже если одна attribution-модель показывает лучшие synthetic-метрики, это **не превращает её в causal estimator**.

Synthetic stress-test показывает:
- как rule-based модели ведут себя при известной скрытой правде;
- насколько вывод чувствителен к модели;
- какие placements можно считать устойчивыми кандидатами на масштабирование.

Для реального incremental ROMI следующий уровень — holdout / randomized promotion / другой causal design.



## 8. Скачать результаты из Colab

После выполнения ноутбука графики лежат в `outputs/charts/`.
В Colab можно скачать их одним ZIP-файлом.


In [ ]:

from pathlib import Path
import shutil
import sys

charts_dir = ROOT / "outputs" / "charts"

if charts_dir.exists():
    archive = shutil.make_archive(
        str(ROOT / "outputs" / "synthetic_attribution_results"),
        "zip",
        root_dir=str(charts_dir)
    )
    print("Archive created:", archive)

    try:
        from google.colab import files
        files.download(archive)
    except Exception:
        print("Not running in Colab — archive stays on disk.")
else:
    print("Run the experiment first: outputs/charts does not exist yet.")
